In [ ]:
import numpy as np
import pandas as pd
import warnings

from tqdm import TqdmWarning
# from extinction import fitzpatrick99
from datetime import datetime
from pathlib import Path
from typing import Literal, Tuple, Dict, Optional, Union, Iterable, Self
from scipy.optimize import curve_fit
from scipy.stats import linregress
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion, Pipeline


warnings.filterwarnings("ignore", category=TqdmWarning)

print("Hello world")

In [ ]:
EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [ ]:
def ingest_dfs(type: Literal["train", "test"]):
    log_df = pd.read_csv(f"{__dirpath}/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"{__dirpath}/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"{__dirpath}/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"{__dirpath}/{split}/{type}_flc.parquet")


__dirpath = Path("../artifacts/kaggle")

In [ ]:
ingest_dfs("train")
ingest_dfs("test")

### Feature Engineering

In [ ]:
def de_extinct(log_df: pd.DataFrame, flc_df: pd.DataFrame):
    EXTINCTION_COEFFS = {
        "u": 4.81,
        "g": 3.64,
        "r": 2.70,
        "i": 2.06,
        "z": 1.58,
        "y": 1.31
    }

    ebv = flc_df["object_id"].map(log_df["EBV"])
    r_λ = flc_df["Filter"].map(EXTINCTION_COEFFS)

    c_λ = np.pow(10, 0.4 * r_λ * ebv)

    flc_df["Flux"] *= c_λ
    flc_df["Flux_err"] *= c_λ


# TODO Consider extinction.fitzpatrick99

In [ ]:
de_extinct(train_log_df, train_flc_df)

train_flc_df

In [ ]:
train_df = TDEFeatureEngineer().generate_features(train_log_df, train_flc_df)
test_df = TDEFeatureEngineer().generate_features(test_log_df, test_flc_df)

train_df["target"] = train_log_df["target"]

train_df

In [ ]:
predictor = TabularPredictor(path = f"../AutogluonModels/ag-{now()}", problem_type="binary", label="target", eval_metric="f1").fit(train_df, presets = "extreme", time_limit=600)

prediction_df = pd.DataFrame({
    "object_id": test_df.index,
    "prediction": predictor.predict(test_df),
})

prediction_df

In [ ]:
__dirpath = Path("../artifacts/predictions")
__dirpath.mkdir(parents=True, exist_ok=True)

prediction_df.to_csv(f"{__dirpath}/submission-{now()}.csv", index=False)